In [39]:
#preapring datasets

import pandas as pd

df=pd.read_csv("test.tsv",sep='\t',header=None)
# print(df.head())

# df.head()
columns = [
    "id",
    "label",
    "statement",
    "subject",
    "speaker",
    "job_title",
    "state",
    "party",
    "barely_true_counts",
    "false_counts",
    "half_true_counts",
    "mostly_true_counts",
    "pants_on_fire_counts",
    "context"
]
df.columns=columns
df.columns
# print(df.isna().sum())
print(df.shape)
df=df[['statement','label']]
df.dropna(inplace=True)
# print(df.isna().sum())
def convert_label(x):
    if x in ["true", "mostly-true"]:
        return 1
    else:
        return 0

df["label"] = df["label"].apply(convert_label)
print(df.shape)
df['label'].value_counts()
print(df['statement'].head())


(1267, 14)
(1267, 2)
0    Building a wall on the U.S.-Mexico border will...
1    Wisconsin is on pace to double the number of l...
2    Says John McCain has done nothing to help the ...
3    Suzanne Bonamici supports a plan that will cut...
4    When asked by a reporter whether hes at the ce...
Name: statement, dtype: object


In [40]:
#NLP preprocessing

import re
import string

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["statement"] = df["statement"].apply(clean_text)
print(df['statement'].head())


from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["statement"], df["label"], test_size=0.2, random_state=42
)
#tokenIZXATIONS

from tensorflow.keras.preprocessing.text import Tokenizer

vocab_size = 5000

tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
print(X_train_seq)

0    building a wall on the usmexico border will ta...
1    wisconsin is on pace to double the number of l...
2    says john mccain has done nothing to help the ...
3    suzanne bonamici supports a plan that will cut...
4    when asked by a reporter whether hes at the ce...
Name: statement, dtype: object
[[12, 219, 1004, 267, 736, 8, 2, 1, 150, 1005], [7, 1559, 737, 1560, 1561, 21, 580, 5, 738, 29, 1562, 6, 1, 739, 124, 1563, 10, 406, 4, 34, 1564], [7, 201, 151, 239, 2, 124, 480, 29, 5, 581, 10, 1565, 240, 1006], [61, 1, 96, 8, 1, 87, 294, 31, 169, 740, 22, 97, 15, 4, 407, 1, 582, 114, 42, 295, 1566, 4, 1, 99, 96], [53, 25, 6, 152, 104, 13, 741, 62, 1567, 12, 1, 187, 170, 1568, 4, 1007, 241, 1569], [742, 481, 268, 8, 1570, 1, 408, 26, 6, 1, 19, 583, 26], [7, 1, 743, 3, 36, 1571, 409, 744, 5, 51, 1572, 4, 584, 1008, 45, 585, 745, 1573, 153, 65, 1009, 202, 1, 1574, 203], [7, 1010, 66, 154, 5, 1575, 8, 410, 115, 1010, 66, 154, 116], [746, 747, 155, 1, 87, 125, 748, 156, 2, 1, 157, 586, 34

In [44]:
embedding_index = {}

with open("glove.6B.100d.txt", encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = list(map(float, values[1:]))
        embedding_index[word] = vector

import numpy as np

embedding_dim = 100
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if i < vocab_size:
        vector = embedding_index.get(word)
        if vector is not None:
            embedding_matrix[i] = vector
print(embedding_matrix[1])

[-0.038194 -0.24487   0.72812  -0.39961   0.083172  0.043953 -0.39141
  0.3344   -0.57545   0.087459  0.28787  -0.06731   0.30906  -0.26384
 -0.13231  -0.20757   0.33395  -0.33848  -0.31743  -0.48336   0.1464
 -0.37304   0.34577   0.052041  0.44946  -0.46971   0.02628  -0.54155
 -0.15518  -0.14107  -0.039722  0.28277   0.14393   0.23464  -0.31021
  0.086173  0.20397   0.52624   0.17164  -0.082378 -0.71787  -0.41531
  0.20335  -0.12763   0.41367   0.55187   0.57908  -0.33477  -0.36559
 -0.54857  -0.062892  0.26584   0.30205   0.99775  -0.80481  -3.0243
  0.01254  -0.36942   2.2167    0.72201  -0.24978   0.92136   0.034514
  0.46745   1.1079   -0.19358  -0.074575  0.23353  -0.052062 -0.22044
  0.057162 -0.15806  -0.30798  -0.41625   0.37972   0.15006  -0.53212
 -0.2055   -1.2526    0.071624  0.70565   0.49744  -0.42063   0.26148
 -1.538    -0.30223  -0.073438 -0.28312   0.37104  -0.25217   0.016215
 -0.017099 -0.38984   0.87424  -0.72569  -0.51058  -0.52028  -0.1459
  0.8278    0.27062 ]